In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col
import pyspark.sql.types as T

In [0]:
df = spark.read.table('olist_dataset.bronze.orders')
display(df)

In [0]:
quarter_df = df.withColumn('quarter', F.concat(F.lit('Q'),F.quarter(col('purchase_time'))))\
            .withColumn('year',F.year(col('purchase_time')))

display(quarter_df)

In [0]:
delivered_data = quarter_df.filter(col('status') == 'delivered').select(
    "order_id",
    "customer_id",
    "status",
    "purchase_time",
    "approved_time",
    "delivered_carrier_date",
    "customer_delivery_date",
    "customer_delivery_est_date",
    "quarter",
    "year"
)
un_delivered_data = quarter_df.filter(col('status') != 'delivered').select(
    "order_id",
    "customer_id",
    "status",
    "purchase_time",
    "approved_time",
    "delivered_carrier_date",
    "customer_delivery_date",
    "customer_delivery_est_date",
    "quarter",
    "year"
)


In [0]:
delivered_data.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable('olist_dataset.silver.delivered_orders')

In [0]:
un_delivered_data.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable('olist_dataset.silver.undelivered_orders')